# Flu Shot Learning - Notebook 13: leakage-free two-stage stacking

Notebook 12 reduced the public score to 0.8621, so this experiment changes the architecture instead of adding more folds. Stage 1 trains CatBoost and LightGBM on the original respondent features. Stage 2 trains target-specific CatBoost meta-models on the original features plus leakage-free out-of-fold base predictions, including the other vaccine's probability. The final blend is selected on OOF predictions only.

This is designed for the competition's mean ROC AUC. Run all cells from top to bottom. The final output is `submission_notebook13_stacked.csv`.


In [1]:
pip install iterative-stratification

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 2026
N_SPLITS = 5
TARGETS = ['h1n1_vaccine', 'seasonal_vaccine']
BASE_MODELS = ['catboost', 'lightgbm']

def find_data_dir():
    candidates = [Path('/kaggle/input/datasets/barshamishra19/flu-shot-learning-dataset'),]
    for candidate in candidates:
        if (candidate / 'training_set_features.csv').exists():
            return candidate
    raise FileNotFoundError('Set DATA_DIR to the folder containing the competition CSV files.')

DATA_DIR = find_data_dir()
train_features = pd.read_csv(DATA_DIR / 'training_set_features.csv')
train_labels = pd.read_csv(DATA_DIR / 'training_set_labels.csv')
test_features = pd.read_csv(DATA_DIR / 'test_set_features.csv')
assert train_features['respondent_id'].is_unique and test_features['respondent_id'].is_unique
assert train_features['respondent_id'].equals(train_labels['respondent_id'])
assert set(train_features['respondent_id']).isdisjoint(test_features['respondent_id'])
y = train_labels[TARGETS].astype('int8')
print(f'Data: {DATA_DIR.resolve()} | train={train_features.shape} | test={test_features.shape}')
display(y.mean().rename('positive_rate').to_frame())


Data: /kaggle/input/datasets/barshamishra19/flu-shot-learning-dataset | train=(26707, 36) | test=(26708, 36)


,positive_rate
h1n1_vaccine,0.212454
seasonal_vaccine,0.465608


In [3]:
def engineer_features(frame):
    result = frame.drop(columns=['respondent_id'], errors='ignore').copy()
    behavior = [c for c in result if c.startswith('behavioral_')]
    opinions = [c for c in result if c.startswith('opinion_')]
    result['missing_count'] = result.isna().sum(axis=1)
    result['missing_fraction'] = result.isna().mean(axis=1)
    result['behavior_score'] = result[behavior].sum(axis=1, min_count=1)
    result['behavior_available'] = result[behavior].notna().sum(axis=1)
    result['opinion_score'] = result[opinions].mean(axis=1)
    result['opinion_std'] = result[opinions].std(axis=1)
    result['doctor_recommendation_count'] = result[['doctor_recc_h1n1', 'doctor_recc_seasonal']].sum(axis=1, min_count=1)
    result['recommendation_gap'] = result['doctor_recc_seasonal'] - result['doctor_recc_h1n1']
    result['health_access_score'] = result[['health_insurance', 'health_worker']].sum(axis=1, min_count=1)
    result['doctor_behavior_alignment'] = result['doctor_recommendation_count'] * result['behavior_score']
    result['doctor_opinion_alignment'] = result['doctor_recommendation_count'] * result['opinion_score']
    result['health_worker_insured'] = result['health_worker'] * result['health_insurance']
    result['age_education'] = result['age_group'].astype('string') + '_' + result['education'].astype('string')
    result['doctor_pair'] = result['doctor_recc_h1n1'].astype('string') + '_' + result['doctor_recc_seasonal'].astype('string')
    ordinal_maps = {
        'age_group': {'18 - 34 Years': 1, '35 - 44 Years': 2, '45 - 54 Years': 3, '55 - 64 Years': 4, '65+ Years': 5},
        'education': {'< 12 Years': 1, '12 Years': 2, 'Some College': 3, 'College Graduate': 4},
        'income_poverty': {'Below Poverty': 1, '<= $75,000, Above Poverty': 2, '> $75,000': 3},
    }
    for column, mapping in ordinal_maps.items():
        result[f'{column}_ordinal'] = result[column].map(mapping)
    return result

X = engineer_features(train_features)
X_test = engineer_features(test_features)
categorical_columns = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
for column in categorical_columns:
    X[column] = X[column].fillna('__MISSING__').astype(str)
    X_test[column] = X_test[column].fillna('__MISSING__').astype(str)
cat_feature_indices = [X.columns.get_loc(c) for c in categorical_columns]
X_lgb = X.copy(); X_test_lgb = X_test.copy()
for column in categorical_columns:
    levels = pd.concat([X_lgb[column], X_test_lgb[column]], ignore_index=True).unique()
    dtype = pd.api.types.CategoricalDtype(categories=levels)
    X_lgb[column] = X_lgb[column].astype(dtype); X_test_lgb[column] = X_test_lgb[column].astype(dtype)
print(f'Engineered features: {X.shape[1]} | categorical: {len(categorical_columns)}')


Engineered features: 52 | categorical: 14


In [4]:
folds = list(MultilabelStratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(X, y))

def make_catboost(target, seed, meta=False):
    if meta:
        settings = {'h1n1_vaccine': (5, 1100, 0.035, 12), 'seasonal_vaccine': (5, 1300, 0.03, 14)}[target]
    else:
        settings = {'h1n1_vaccine': (7, 2200, 0.03, 9), 'seasonal_vaccine': (6, 2400, 0.028, 11)}[target]
    depth, iterations, learning_rate, l2_leaf_reg = settings
    return CatBoostClassifier(depth=depth, iterations=iterations, learning_rate=learning_rate, l2_leaf_reg=l2_leaf_reg, loss_function='Logloss', eval_metric='AUC', random_seed=seed, verbose=False, allow_writing_files=False, thread_count=-1)

def make_lightgbm(target, seed):
    return LGBMClassifier(objective='binary', n_estimators=2800, learning_rate=0.018, num_leaves=24 if target == 'h1n1_vaccine' else 31, min_child_samples=40, subsample=0.9, colsample_bytree=0.9, reg_alpha=0.15, reg_lambda=1.8, random_state=seed, n_jobs=-1, verbosity=-1)


In [5]:
# Stage 1: leakage-free OOF predictions and fold-averaged test predictions.
base_oof = {name: pd.DataFrame(0.0, index=X.index, columns=TARGETS) for name in BASE_MODELS}
base_test = {name: pd.DataFrame(0.0, index=X_test.index, columns=TARGETS) for name in BASE_MODELS}
for fold_number, (train_idx, valid_idx) in enumerate(folds, start=1):
    print(f'Stage 1 fold {fold_number}/{N_SPLITS}')
    for target in TARGETS:
        seed = RANDOM_STATE + fold_number
        cat = make_catboost(target, seed)
        cat.fit(X.iloc[train_idx], y.iloc[train_idx][target], cat_features=cat_feature_indices, eval_set=(X.iloc[valid_idx], y.iloc[valid_idx][target]), use_best_model=True, early_stopping_rounds=160)
        base_oof['catboost'].loc[valid_idx, target] = cat.predict_proba(X.iloc[valid_idx])[:, 1]
        base_test['catboost'][target] += cat.predict_proba(X_test)[:, 1] / N_SPLITS
        lgb = make_lightgbm(target, seed + 100)
        lgb.fit(X_lgb.iloc[train_idx], y.iloc[train_idx][target], eval_set=[(X_lgb.iloc[valid_idx], y.iloc[valid_idx][target])], eval_metric='auc', categorical_feature=categorical_columns, callbacks=[early_stopping(160, verbose=False), log_evaluation(0)])
        base_oof['lightgbm'].loc[valid_idx, target] = lgb.predict_proba(X_lgb.iloc[valid_idx])[:, 1]
        base_test['lightgbm'][target] += lgb.predict_proba(X_test_lgb)[:, 1] / N_SPLITS
base_scores = pd.DataFrame({name: [roc_auc_score(y[target], base_oof[name][target]) for target in TARGETS] for name in BASE_MODELS}, index=TARGETS)
base_scores['mean'] = base_scores.mean(axis=1)
display(base_scores)


Stage 1 fold 1/5
Stage 1 fold 2/5
Stage 1 fold 3/5
Stage 1 fold 4/5
Stage 1 fold 5/5


,catboost,lightgbm,mean
h1n1_vaccine,0.871870,0.870746,0.871308
seasonal_vaccine,0.864602,0.863256,0.863929


In [6]:
# Stage 2: add the OOF base predictions and cross-target predictions as meta-features.
def add_meta_features(frame, predictions, include_target_predictions=True):
    result = frame.copy()
    for model_name in BASE_MODELS:
        for target in TARGETS:
            result[f'{model_name}_{target}_base'] = predictions[model_name][target].to_numpy()
    if include_target_predictions:
        result['base_h1n1_mean'] = np.mean([predictions[name]['h1n1_vaccine'] for name in BASE_MODELS], axis=0)
        result['base_seasonal_mean'] = np.mean([predictions[name]['seasonal_vaccine'] for name in BASE_MODELS], axis=0)
        result['base_target_gap'] = result['base_seasonal_mean'] - result['base_h1n1_mean']
    return result

meta_X = add_meta_features(X, base_oof)
meta_test = add_meta_features(X_test, base_test)
meta_categorical = meta_X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
for column in meta_categorical:
    meta_X[column] = meta_X[column].fillna('__MISSING__').astype(str)
    meta_test[column] = meta_test[column].fillna('__MISSING__').astype(str)
meta_cat_indices = [meta_X.columns.get_loc(c) for c in meta_categorical]


In [7]:
# Evaluate the stacker with the same folds. The meta-features are OOF for every training row.
meta_oof = pd.DataFrame(0.0, index=X.index, columns=TARGETS)
meta_test_pred = pd.DataFrame(0.0, index=X_test.index, columns=TARGETS)
for fold_number, (train_idx, valid_idx) in enumerate(folds, start=1):
    for target in TARGETS:
        meta = make_catboost(target, RANDOM_STATE + 500 + fold_number, meta=True)
        meta.fit(meta_X.iloc[train_idx], y.iloc[train_idx][target], cat_features=meta_cat_indices, eval_set=(meta_X.iloc[valid_idx], y.iloc[valid_idx][target]), use_best_model=True, early_stopping_rounds=120)
        meta_oof.loc[valid_idx, target] = meta.predict_proba(meta_X.iloc[valid_idx])[:, 1]
        meta_test_pred[target] += meta.predict_proba(meta_test)[:, 1] / N_SPLITS
meta_scores = pd.Series({target: roc_auc_score(y[target], meta_oof[target]) for target in TARGETS})
print('Stage 2 OOF scores:'); display(meta_scores.to_frame('meta_auc'))


Stage 2 OOF scores:


,meta_auc
h1n1_vaccine,0.868325
seasonal_vaccine,0.858624


In [8]:
# Select a conservative target-specific base/meta blend on OOF predictions only.
selected = {}
for target in TARGETS:
    candidates = []
    for meta_weight in np.arange(0.0, 1.01, 0.05):
        base = 0.5 * base_oof['catboost'][target] + 0.5 * base_oof['lightgbm'][target]
        prediction = (1.0 - meta_weight) * base + meta_weight * meta_oof[target]
        score = roc_auc_score(y[target], prediction)
        candidates.append((score, float(meta_weight)))
    score, meta_weight = max(candidates)
    selected[target] = {'meta_weight': meta_weight, 'oof_auc': score}
    print(target, selected[target])
print('Selected mean OOF ROC AUC:', np.mean([v['oof_auc'] for v in selected.values()]))


h1n1_vaccine {'meta_weight': 0.0, 'oof_auc': np.float64(0.8727166087311207)}
seasonal_vaccine {'meta_weight': 0.0, 'oof_auc': np.float64(0.8651962627186031)}
Selected mean OOF ROC AUC: 0.8689564357248618


In [9]:
final_predictions = pd.DataFrame(index=X_test.index, columns=TARGETS, dtype=float)
for target in TARGETS:
    base_test_mean = 0.5 * base_test['catboost'][target] + 0.5 * base_test['lightgbm'][target]
    weight = selected[target]['meta_weight']
    final_predictions[target] = np.clip((1.0 - weight) * base_test_mean + weight * meta_test_pred[target], 0.0, 1.0)
submission = pd.DataFrame({'respondent_id': test_features['respondent_id'], **{target: final_predictions[target] for target in TARGETS}})
assert list(submission.columns) == ['respondent_id', *TARGETS]
assert submission['respondent_id'].equals(test_features['respondent_id'])
assert submission[TARGETS].notna().all().all()
output_dir = Path.cwd() / 'outputs'; output_dir.mkdir(parents=True, exist_ok=True)
submission_path = output_dir / 'submission_notebook13_stacked.csv'
oof_path = output_dir / 'oof_notebook13_stacked.csv'
submission.to_csv(submission_path, index=False)
oof = pd.DataFrame({'respondent_id': train_features['respondent_id']})
for target in TARGETS:
    oof[f'base_catboost_{target}'] = base_oof['catboost'][target]
    oof[f'base_lightgbm_{target}'] = base_oof['lightgbm'][target]
    oof[f'meta_{target}'] = meta_oof[target]
oof.to_csv(oof_path, index=False)
print(f'Saved submission: {submission_path.resolve()}')
print(f'Saved OOF predictions: {oof_path.resolve()}')
display(submission.head())


Saved submission: /kaggle/working/outputs/submission_notebook13_stacked.csv
Saved OOF predictions: /kaggle/working/outputs/oof_notebook13_stacked.csv


,respondent_id,h1n1_vaccine,seasonal_vaccine
0,26707,0.134904,0.231301
1,26708,0.030433,0.035794
2,26709,0.222239,0.655057
3,26710,0.650855,0.878206
4,26711,0.250478,0.492304
